Detecting Engine Faults using GNN

In [1]:
import pandas as pd
import torch

engine_faultdb = 'EngineFaultDB/EngineFaultDB_Final.csv'
df = pd.read_csv(engine_faultdb)

Preprocess: Normalize all columns (StandardScaler) because RPM (1000+) and Lambda (1.0) have vastly different scales. GNNs fail without this.

In [2]:
from torch.utils.data import Dataset 
from torch_geometric.data import Data

class EngineGraphDataset(Dataset):
    def __init__(self, df, labels, edge_index):
        self.x = torch.tensor(df.values, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.edge_index = edge_index

    def len(self):
        return len(self.x)
    
    def __len__(self):
        return self.len()
    
    def __getitem__(self, idx):
        x = self.x[idx].unsqueeze(1)
        y = self.labels[idx]
        return Data(x=x, edge_index=self.edge_index, y=y)

/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np

def create_graph_nodes(df):
  corr_matrix = df.corr()

  threshold = 0.6

  mask = (corr_matrix > threshold) & (np.eye(len(corr_matrix)) == 0)

  src, dst = np.where(mask)

  edges = torch.tensor([src, dst], dtype=torch.long)
  return edges

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import DataLoader

train_data, test_data = train_test_split(df, test_size=0.2, random_state=42, shuffle=True) # Shuffle=True for training!

y_train = train_data['Fault']
X_train = train_data.drop(columns=['Fault'])

y_test = test_data['Fault']
X_test = test_data.drop(columns=['Fault'])

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

edge_index = create_graph_nodes(X_train_scaled)

train_labels_multi_class = y_train.values.astype(int)
train_dataset = EngineGraphDataset(X_train_scaled, train_labels_multi_class, edge_index)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_labels_multi_class = y_test.values.astype(int)
test_dataset = EngineGraphDataset(X_test_scaled, test_labels_multi_class, edge_index)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_78429/681140456.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  edges = torch.tensor([src, dst], dtype=torch.long)
/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_78429/1585582152.py:21: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_78429/1585582152.py:25: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


Graph Construction: Create an adjacency matrix based on correlation (e.g., TPS vs RPM, AFR vs Lambda).

Compute Correlation Matrix

In [5]:
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F

class GNN(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch) 
        x = self.lin(x)
        return x

Train: Train a GCN to classify the "Fault" label.

In [7]:
from tqdm.auto import tqdm

model = GNN(num_node_features=1, hidden_channels=32, num_classes=4)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

epochs = 50
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch}", unit="batch")
    
    for batch in loop:
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} finished. Avg Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 700/700 [00:03<00:00, 203.88batch/s, loss=1.11] 


Epoch 1 finished. Avg Loss: 1.2001


Epoch 2: 100%|██████████| 700/700 [00:03<00:00, 209.76batch/s, loss=1.04] 


Epoch 2 finished. Avg Loss: 1.0364


Epoch 3: 100%|██████████| 700/700 [00:03<00:00, 216.07batch/s, loss=0.892]


...

Epoch 48 finished. Avg Loss: 0.6034


Epoch 49: 100%|██████████| 700/700 [00:03<00:00, 178.12batch/s, loss=0.583]


Epoch 49 finished. Avg Loss: 0.6056


Epoch 50: 100%|██████████| 700/700 [00:03<00:00, 194.44batch/s, loss=0.676]

Epoch 50 finished. Avg Loss: 0.6114


In [8]:
import torch

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        out = model(batch.x, batch.edge_index, batch.batch)
      
        preds = out.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, digits=4, target_names=["Fault0", "Fault1", "Fault2", "Fault3"]))


[[2967  143  111    8]
 [ 147 1968   42   37]
 [ 127  232 2199  462]
 [ 117  206 1995  439]]
              precision    recall  f1-score   support

      Fault0     0.8836    0.9189    0.9009      3229
      Fault1     0.7721    0.8970    0.8299      2194
      Fault2     0.5059    0.7281    0.5970      3020
      Fault3     0.4641    0.1592    0.2371      2757

    accuracy                         0.6762     11200
   macro avg     0.6564    0.6758    0.6412     11200
weighted avg     0.6566    0.6762    0.6416     11200



In [10]:
class EngineTabularDataset(Dataset):
    def __init__(self, X_df, y_np):
        self.X = torch.tensor(X_df.values, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.long)

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = EngineTabularDataset(X_train_scaled, train_labels_multi_class)
test_ds  = EngineTabularDataset(X_test_scaled,  test_labels_multi_class)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, drop_last=False)

/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_78429/2180761564.py:15: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=False)
/var/folders/02/q215g0zs37l9h3w3x7ntgygw0000gn/T/ipykernel_78429/2180761564.py:16: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, drop_last=False)


In [11]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, in_dim, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)  # <- 4 logits
        )
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = MLP(in_dim=X_train_scaled.shape[1], num_classes=4).to(device)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 50
for epoch in range(1, epochs + 1):
    model.train()
    losses = []

    loop = tqdm(train_loader, desc=f"Epoch {epoch}", unit="batch")
    for xb, yb in loop:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        loop.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{np.mean(losses):.4f}")

Epoch 50: 100%|██████████| 175/175 [00:01<00:00, 168.11batch/s, avg_loss=0.3483, loss=0.3428]


In [14]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, digits=4, target_names=["Fault0", "Fault1", "Fault2", "Fault3"]))

[[3229    0    0    0]
 [   0 2194    0    0]
 [   0    0 2809  211]
 [   0    0 2556  201]]
              precision    recall  f1-score   support

      Fault0     1.0000    1.0000    1.0000      3229
      Fault1     1.0000    1.0000    1.0000      2194
      Fault2     0.5236    0.9301    0.6700      3020
      Fault3     0.4879    0.0729    0.1269      2757

    accuracy                         0.7529     11200
   macro avg     0.7529    0.7508    0.6992     11200
weighted avg     0.7455    0.7529    0.6961     11200

